# CordiaAI — Step 18: Coronary CT branch (ImageCAS + 3D Swin-UNETR) → coronary tree for the twin

Segments the **coronary arteries** in a CT angiogram with a 3D Swin-UNETR, then turns the mask into a 3D
mesh of the patient's own coronary tree for the digital twin. ImageCAS has no per-patient disease label,
so this branch is about **anatomy** (what the twin shows), not a disease score.

**Before running:** select a T4 (or better) GPU. The data comes from the authors' Kaggle upload
(`xiaoweixumedicalai/imagecas`): the next cell downloads it to Colab's local disk with your Kaggle API token
(kaggle.com -> Settings -> API -> Create New Token -> `kaggle.json`). It is too big for a free Drive.
If you already have the NIfTI files in `MyDrive/cordia/ImageCAS/`, skip that cell.

**Runtime budget:** 3D training is heavy. `N_TRAIN` below uses 150 scans so a run fits in one free Colab
session; raise it on a bigger GPU. **Split:** by scan, fixed seed: 30 test, 20 validation, the rest (up to `N_TRAIN`) train.

**Same code as the app:** the cells marked `%%writefile` are copies of this project's own modules
(`src/models/imaging/`, `src/imaging/`), generated from them, so the model trained here is exactly
the one `mobile_app/backend/main.py` loads.

In [ ]:
!pip install -q monai einops nibabel scikit-image
import torch
print('CUDA:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
import os
for d in ['src/models/imaging', 'src/imaging']:
    os.makedirs(d, exist_ok=True)
for f in ['src/__init__.py', 'src/models/__init__.py', 'src/models/imaging/__init__.py', 'src/imaging/__init__.py']:
    open(f, 'a').close()

In [ ]:
%%writefile src/models/imaging/cardiac_seg.py
"""MRI and CT branches: Swin-UNETR segmentation (transformer encoder + U-Net decoder).

A plain ViT gives one label per image; the digital twin needs the outline of
each heart chamber, pixel by pixel. Swin-UNETR (Hatamizadeh et al., 2022)
keeps a transformer as the encoder and adds a convolutional decoder that
produces a full-resolution mask, so it is still a vision-transformer model.
Implementation from MONAI.

- MRI (ACDC): 2D, one short-axis slice at a time. ACDC slices are 5-10 mm
  apart, so per-slice 2D segmentation is the standard choice. Labels follow
  ACDC: 1 = RV cavity, 2 = LV myocardium, 3 = LV cavity.
  Trained in `notebooks/step17_mri_acdc_colab.ipynb`.
- CT (ImageCAS): 3D patches, binary coronary artery mask.
  Trained in `notebooks/step18_ct_imagecas_colab.ipynb`.
"""
from __future__ import annotations

import inspect

from monai.networks.nets import SwinUNETR

ACDC_LABELS = {0: "background", 1: "RV cavity", 2: "LV myocardium", 3: "LV cavity"}
ACDC_GROUPS = ["NOR", "MINF", "DCM", "HCM", "RV"]
ACDC_GROUP_NAMES = {
    "NOR": "Normal",
    "MINF": "Previous myocardial infarction",
    "DCM": "Dilated cardiomyopathy",
    "HCM": "Hypertrophic cardiomyopathy",
    "RV": "Abnormal right ventricle",
}

MRI_SLICE_SIZE = 224  # divisible by 32, as Swin-UNETR requires
CT_PATCH_SIZE = (96, 96, 96)


def _swin_unetr(img_size, **kwargs) -> SwinUNETR:
    # MONAI deprecated and then removed the img_size argument; pass it only
    # to versions that still take it.
    if "img_size" in inspect.signature(SwinUNETR.__init__).parameters:
        kwargs["img_size"] = img_size
    return SwinUNETR(**kwargs)


def build_mri_segmenter(feature_size: int = 24) -> SwinUNETR:
    return _swin_unetr(
        (MRI_SLICE_SIZE, MRI_SLICE_SIZE), in_channels=1, out_channels=len(ACDC_LABELS),
        feature_size=feature_size, spatial_dims=2,
    )


def build_ct_segmenter(feature_size: int = 48, use_checkpoint: bool = True) -> SwinUNETR:
    return _swin_unetr(
        CT_PATCH_SIZE, in_channels=1, out_channels=2, feature_size=feature_size,
        spatial_dims=3, use_checkpoint=use_checkpoint,
    )

In [ ]:
%%writefile src/imaging/ct.py
"""Coronary CT angiography pipeline (ImageCAS), shared by training and the app.

CT scans differ in voxel size from scanner to scanner, so every scan is
resampled to one isotropic spacing before the model sees it, and the
intensity window is fixed in Hounsfield units (contrast-filled coronary
arteries sit around 300-500 HU). The same transforms run in
notebooks/step18_ct_imagecas_colab.ipynb and in the app.
"""
from __future__ import annotations

import numpy as np
import torch
from monai import transforms as T

CT_SPACING = (0.8, 0.8, 0.8)  # mm
HU_MIN, HU_MAX = -200.0, 800.0
PATCH = (96, 96, 96)


def _base(keys, label: bool):
    modes = ("bilinear", "nearest") if label else ("bilinear",)
    return [
        T.LoadImaged(keys=keys),
        T.EnsureChannelFirstd(keys=keys),
        T.Orientationd(keys=keys, axcodes="RAS"),
        T.Spacingd(keys=keys, pixdim=CT_SPACING, mode=modes),
        T.ScaleIntensityRanged(keys=["image"], a_min=HU_MIN, a_max=HU_MAX, b_min=0.0, b_max=1.0, clip=True),
    ]


def train_transforms(samples_per_scan: int = 2):
    keys = ["image", "label"]
    return T.Compose(_base(keys, True) + [
        T.CropForegroundd(keys=keys, source_key="image"),
        # Coronary arteries are <1% of the volume: sample patches centred on vessel twice as often as background
        T.RandCropByPosNegLabeld(keys=keys, label_key="label", spatial_size=PATCH, pos=2, neg=1,
                                 num_samples=samples_per_scan),
        T.RandFlipd(keys=keys, prob=0.5, spatial_axis=0),
        T.RandScaleIntensityd(keys=["image"], factors=0.1, prob=0.5),
    ])


def val_transforms():
    return T.Compose(_base(["image", "label"], True))


def inference_transforms():
    return T.Compose(_base(["image"], False))


@torch.no_grad()
def segment_ct_file(model: torch.nn.Module, path: str) -> tuple[np.ndarray, tuple]:
    """NIfTI CT file -> (binary coronary mask at CT_SPACING, spacing)."""
    from monai.inferers import sliding_window_inference

    data = inference_transforms()({"image": path})
    x = data["image"].unsqueeze(0).float()
    logits = sliding_window_inference(x, PATCH, sw_batch_size=2, predictor=model, overlap=0.25)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8), CT_SPACING

In [ ]:
%%writefile src/imaging/mesh.py
"""3D meshes of heart structures from segmentation masks, for the digital twin.

Marching cubes turns each labelled structure (LV cavity, myocardium, RV,
coronary tree) into a triangle surface in millimetres. The result is sent
to heart_digital_twin_3d/ as plain JSON (vertex + face arrays a Three.js
BufferGeometry takes directly), or saved as .obj for any 3D viewer.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage
from skimage import measure


def mask_to_mesh(mask: np.ndarray, spacing_mm, step: int = 2, smooth_sigma: float = 1.0) -> dict | None:
    """Binary mask (H, W, D) -> {"vertices": [[x,y,z]...], "faces": [[i,j,k]...]} in mm."""
    if mask.sum() == 0:
        return None
    # Pad so surfaces touching the volume edge still close, then smooth the
    # blocky voxel edges (MRI slices are 5-10 mm thick).
    padded = np.pad(mask.astype(np.float32), 1)
    if smooth_sigma > 0:
        sigma = [smooth_sigma / s for s in spacing_mm]
        smoothed = ndimage.gaussian_filter(padded, sigma=sigma)
        if smoothed.max() > 0.5:  # a structure only a few voxels across can blur away entirely
            padded = smoothed
    try:
        verts, faces, _, _ = measure.marching_cubes(padded, level=0.5, spacing=tuple(spacing_mm), step_size=step)
    except (RuntimeError, ValueError):  # too small or thin to form a surface
        return None
    verts -= np.array(spacing_mm)  # undo the 1-voxel pad
    return {"vertices": np.round(verts, 2).tolist(), "faces": faces.astype(int).tolist()}


def label_meshes(mask: np.ndarray, spacing_mm, labels: dict[int, str], **kwargs) -> dict[str, dict]:
    meshes = {}
    for value, name in labels.items():
        if value == 0:
            continue
        mesh = mask_to_mesh(mask == value, spacing_mm, **kwargs)
        if mesh is not None:
            meshes[name] = mesh
    return meshes


def mesh_to_obj(mesh: dict) -> str:
    lines = [f"v {x} {y} {z}" for x, y, z in mesh["vertices"]]
    lines += [f"f {a + 1} {b + 1} {c + 1}" for a, b, c in mesh["faces"]]
    return "\n".join(lines) + "\n"

## Download ImageCAS from Kaggle
Upload your `kaggle.json` when asked. Skip this cell if the data is already in Drive.

In [ ]:
from google.colab import files
files.upload()  # kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle datasets download -d xiaoweixumedicalai/imagecas -p /content/ImageCAS --unzip
!cd /content/ImageCAS && for z in $(find . -name '*.zip'); do unzip -q -o "$z" -d "$(dirname "$z")" && rm "$z"; done
!find /content/ImageCAS -name '*.nii.gz' | head -4; find /content/ImageCAS -name '*.nii.gz' | wc -l

## Pair each CT with its label file
ImageCAS names files by case number; label files carry `label` in the name. Adjust `IMAGECAS` if your folder differs.

In [ ]:
import glob, os, re
import numpy as np
IMAGECAS = '/content/ImageCAS'  # from the Kaggle cell
if not glob.glob(f'{IMAGECAS}/**/*.nii.gz', recursive=True):
    from google.colab import drive
    drive.mount('/content/drive')
    IMAGECAS = '/content/drive/MyDrive/cordia/ImageCAS'

files = glob.glob(f'{IMAGECAS}/**/*.nii.gz', recursive=True)
case = lambda f: re.search(r'(\d+)', os.path.basename(f)).group(1)
labels = {case(f): f for f in files if 'label' in os.path.basename(f).lower()}
images = {case(f): f for f in files if 'label' not in os.path.basename(f).lower()}
pairs = [{'image': images[c], 'label': labels[c]} for c in sorted(images, key=int) if c in labels]
print(f'{len(pairs)} image/label pairs found')
assert pairs, 'No pairs found: check IMAGECAS and the file names'

rng = np.random.default_rng(42); idx = rng.permutation(len(pairs))
N_TRAIN = 150
n_test, n_val = min(30, len(pairs) // 5), min(20, len(pairs) // 5)
test = [pairs[i] for i in idx[:n_test]]; val = [pairs[i] for i in idx[n_test:n_test + n_val]]
train = [pairs[i] for i in idx[n_test + n_val:n_test + n_val + N_TRAIN]]
print(len(train), len(val), len(test))

In [ ]:
from monai.data import CacheDataset, DataLoader, Dataset
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from src.imaging.ct import train_transforms, val_transforms, PATCH
from src.models.imaging.cardiac_seg import build_ct_segmenter

train_ds = CacheDataset(train, train_transforms(), cache_rate=0.5, num_workers=4)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
val_loader = DataLoader(Dataset(val, val_transforms()), batch_size=1)
test_loader = DataLoader(Dataset(test, val_transforms()), batch_size=1)

model = build_ct_segmenter().to(device)
criterion = DiceCELoss(to_onehot_y=True, softmax=True)
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')

## Train

In [ ]:
dice_metric = DiceMetric(include_background=False)

def evaluate(loader):
    model.eval(); dice_metric.reset()
    with torch.no_grad(), torch.autocast(device, enabled=device == 'cuda'):
        for b in loader:
            logits = sliding_window_inference(b['image'].to(device), PATCH, 2, model, overlap=0.25)
            pred = torch.nn.functional.one_hot(logits.argmax(1), 2).permute(0, 4, 1, 2, 3)
            gt = torch.nn.functional.one_hot(b['label'][:, 0].long().to(device), 2).permute(0, 4, 1, 2, 3)
            dice_metric(pred, gt)
    return dice_metric.aggregate().item()

EPOCHS, best, best_state = 40, 0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    for b in train_loader:
        x, y = b['image'].to(device), b['label'].to(device)
        with torch.autocast(device, enabled=device == 'cuda'):
            loss = criterion(model(x), y)
        optimiser.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimiser); scaler.update()
    if epoch % 5 == 0:
        d = evaluate(val_loader)
        print(f'epoch {epoch} loss {loss.item():.3f} val coronary Dice {d:.3f}')
        if best_state is None or d > best:
            best, best_state = d, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)
print(f'Test coronary Dice: {evaluate(test_loader):.3f}')

## Coronary tree mesh for the twin

In [ ]:
import json
from src.imaging.ct import segment_ct_file, CT_SPACING
from src.imaging.mesh import mask_to_mesh
mask, sp = segment_ct_file(model.eval(), test[0]['image'])
mesh = mask_to_mesh(mask, sp, step=1, smooth_sigma=0.5)
print(len(mesh['vertices']), 'vertices')
json.dump({'case': os.path.basename(test[0]['image']), 'meshes': {'Coronary arteries': mesh}}, open('sample_coronary_mesh.json', 'w'))

torch.save(best_state, 'ct_swin_unetr.pt')
from google.colab import files
for f in ('ct_swin_unetr.pt', 'sample_coronary_mesh.json'):
    files.download(f)  # put them in cordia_ai/checkpoints/